# Lesson 7.1: Chi-Square Tests for Categories

Test whether ad type (video or banner) is related to buying, using the lesson's 200-person table, and measure the size of the link with Cramér's V.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Enter the table of counts

Rows are the two ads and columns are bought or not. A chi-square test works on counts like these, never on percentages. Totals first, so you can see where the expected counts will come from.

In [1]:
import numpy as np
from scipy import stats

# rows: video ad, banner ad    columns: bought, did not buy
observed = np.array([[60, 40],
                     [45, 55]])

print('row totals   :', observed.sum(axis=1))
print('column totals:', observed.sum(axis=0))
print('grand total  :', observed.sum())

row totals   : [100 100]
column totals: [105  95]
grand total  : 200


## 2. Run the chi-square test

chi2_contingency works out the expected counts (what we would see if ad type and buying were unrelated), then the chi-square statistic, degrees of freedom and p-value. correction=False gives the plain formula used in the lesson.

In [2]:
res = stats.chi2_contingency(observed, correction=False)

print('expected counts:')
print(res.expected_freq)
print(f'chi-square = {res.statistic:.3f}')
print(f'df         = {res.dof}')
print(f'p-value    = {res.pvalue:.4f}')

expected counts:
[[52.5 47.5]
 [52.5 47.5]]
chi-square = 4.511
df         = 1
p-value    = 0.0337


## 3. Check it by hand

Expected = row total x column total / grand total. Each cell then adds (observed - expected)^2 / expected, and the four pieces sum to the same chi-square.

In [3]:
rows = observed.sum(axis=1)
cols = observed.sum(axis=0)
n = observed.sum()

expected = np.outer(rows, cols) / n
pieces = (observed - expected) ** 2 / expected

print(pieces.round(3))
print(f'sum of pieces = {pieces.sum():.3f}')

[[1.071 1.184]
 [1.071 1.184]]
sum of pieces = 4.511


## 4. Measure the effect with Cramér's V

A small p-value says the link is probably real, not how big it is. Cramér's V = sqrt(chi-square / (N x (k - 1))), where k is the smaller of the number of rows and columns. It runs from 0 (no link) to 1 (perfect link).

In [4]:
chi2 = res.statistic
k = min(observed.shape)
v = np.sqrt(chi2 / (n * (k - 1)))

print(f'Cramer V = {v:.3f}')
print('bought: video', observed[0, 0] / rows[0], ' banner', observed[1, 0] / rows[1])

Cramer V = 0.150
bought: video 0.6  banner 0.45


## 5. Bonus: goodness of fit

A goodness-of-fit test checks one set of counts against a claimed pattern. Here a die was rolled 120 times; chisquare assumes every face is equally likely unless you give it other expected counts.

In [5]:
rolls = np.array([25, 17, 15, 23, 24, 16])

fit = stats.chisquare(rolls)

print(f'expected per face = {rolls.sum() / 6:.0f}')
print(f'chi-square = {fit.statistic:.3f}, df = {len(rolls) - 1}')
print(f'p-value    = {fit.pvalue:.4f}')

expected per face = 20
chi-square = 5.000, df = 5
p-value    = 0.4159


## Try this

- Change the banner row to 50 and 50 (so both ads convert 50 to 60 percent) and predict what happens to chi-square and Cramer's V.
- Multiply every count by 2 (400 people, same percentages). Which changes, the p-value or Cramer's V, and why?

## The same idea in R, SPSS and Stata

**R**

```r
ads <- matrix(c(60, 45, 40, 55), nrow = 2,
              dimnames = list(ad = c("video", "banner"), bought = c("yes", "no")))
test <- chisq.test(ads, correct = FALSE)
test
test$expected
sqrt(test$statistic / (sum(ads) * (min(dim(ads)) - 1)))
chisq.test(c(25, 17, 15, 23, 24, 16))
```

**SPSS** (Analyze > Descriptive Statistics > Crosstabs (Statistics: Chi-square, Phi and Cramer's V; Cells: Expected))

```
DATA LIST FREE / ad buy count.
BEGIN DATA
1 1 60  1 2 40  2 1 45  2 2 55
END DATA.
VALUE LABELS ad 1 'video' 2 'banner' / buy 1 'bought' 2 'did not buy'.
WEIGHT BY count.
CROSSTABS
  /TABLES=ad BY buy
  /STATISTICS=CHISQ PHI
  /CELLS=COUNT EXPECTED.
```

**Stata**

```
clear
input ad buy freq
1 1 60
1 2 40
2 1 45
2 2 55
end
tabulate ad buy [fweight = freq], chi2 V expected
```